In [1]:
from ultralytics import YOLO
import os
import cv2
import torch
import math

In [45]:
def sliding_window_prediction(image, model, conf_threshold=0):
    height, width, _ = image.shape
    tile_size = 640
    stride = 420

    num_windows_y = (height - tile_size) // stride + 1
    num_windows_x = (width - tile_size) // stride + 1

    all_boxes, all_scores, all_classes = [], [], []

    for i in range(num_windows_y):
        for j in range(num_windows_x):
            y_start, x_start = i * stride, j * stride
            y_end, x_end = y_start + tile_size, x_start + tile_size

            window = image[y_start:y_end, x_start:x_end]
            pad_bottom, pad_right = max(0, y_end - height), max(0, x_end - width)
            window_padded = cv2.copyMakeBorder(
                window, 0, pad_bottom, 0, pad_right, cv2.BORDER_CONSTANT, value=(0, 0, 0)
            )

            window_rgb = cv2.cvtColor(window_padded, cv2.COLOR_BGR2RGB)
            window_tensor = (
                torch.tensor(window_rgb / 255.0)
                .permute(2, 0, 1)
                .unsqueeze(0)
                .float()
                .to(model.device)
            )

            results = model(window_tensor, verbose=False, augment=False)
            predictions = results[0].boxes

            valid = predictions.conf > conf_threshold
            boxes = predictions.xyxy[valid]
            scores = predictions.conf[valid]
            classes = predictions.cls[valid]
            
            #shifting the coordinates back to full image
            boxes[:, [0, 2]] += x_start
            boxes[:, [1, 3]] += y_start

            all_boxes.append(boxes)
            all_scores.append(scores)
            all_classes.append(classes)

    if len(all_boxes) == 0:
        return torch.zeros((0, 4), device=model.device), torch.zeros((0,), device=model.device), torch.zeros((0,), device=model.device)

    return (
        torch.cat(all_boxes, dim=0),
        torch.cat(all_scores, dim=0),
        torch.cat(all_classes, dim=0),
    )

def nms(boxes, scores, classes, iou_threshold=0.5, device="cuda"):
    if boxes.numel() == 0:
        return boxes, scores, classes
    keep = torch.ops.torchvision.nms(
        boxes.to(device), scores.to(device), iou_threshold
    )
    return boxes[keep], scores[keep], classes[keep]

def filter_mostly_contained_boxes(boxes, scores, classes, threshold=0.5):
    if boxes.numel() == 0:
        return boxes, scores, classes

    boxes = boxes.float()
    scores = scores.float()

    areas = (boxes[:, 2] - boxes[:, 0]).clamp(min=0) * (boxes[:, 3] - boxes[:, 1]).clamp(min=0)

    xx1 = torch.max(boxes[:, None, 0], boxes[None, :, 0])
    yy1 = torch.max(boxes[:, None, 1], boxes[None, :, 1])
    xx2 = torch.min(boxes[:, None, 2], boxes[None, :, 2])
    yy2 = torch.min(boxes[:, None, 3], boxes[None, :, 3])

    w = (xx2 - xx1).clamp(min=0)
    h = (yy2 - yy1).clamp(min=0)
    inter = w * h

    containment_ratio = inter / (areas[:, None] + 1e-6)

    # j can suppress i if score[j] >= score[i]
    higher_or_equal_conf_mask = scores[None, :] >= scores[:, None]

    containment_mask = (containment_ratio >= threshold) & higher_or_equal_conf_mask

    # remove self-comparisons
    containment_mask.fill_diagonal_(False)

    mostly_contained = containment_mask.any(dim=1)
    keep = ~mostly_contained

    return boxes[keep], scores[keep], classes[keep]

In [57]:
def get_labels_per_tile_tensor(image, boxes, classes, scores, tile_size=640, stride=440, min_inside_ratio=0.8):
    padded_img, orig_w, orig_h = pad_to_multiple(image, tile_size=tile_size)
    h, w = padded_img.shape[:2]

    tiles_data = []
    for y in range(0, h - tile_size + 1, stride):
        for x in range(0, w - tile_size + 1, stride):
            tile_box = (x, y, x + tile_size, y + tile_size)
            tile_labels = save_tile_labels_to_tensor(boxes, classes, scores, tile_box, min_inside_ratio)
            tiles_data.append(tile_labels)
    return tiles_data


def save_tile_labels_to_tensor(boxes, classes, scores, tile_box, min_inside_ratio=0.8):
    """
    boxes: [N,4] tensor in absolute coordinates (xmin, ymin, xmax, ymax)
    returns: [M,6] tensor per tile with absolute clipped boxes: [class, xmin, ymin, xmax, ymax, score]
    """
    x_tile, y_tile, x_tile2, y_tile2 = tile_box

    # Compute box areas and non-zero mask
    box_areas = (boxes[:,2] - boxes[:,0]) * (boxes[:,3] - boxes[:,1])
    non_zero_mask = box_areas > 0
    boxes, classes, scores, box_areas = boxes[non_zero_mask], classes[non_zero_mask], scores[non_zero_mask], box_areas[non_zero_mask]
    
    # Intersection area with tile
    inter_areas = compute_intersection_area_tensor(torch.tensor(tile_box), boxes)
    inside_ratio = inter_areas / box_areas
    mask = inside_ratio >= min_inside_ratio

    boxes, classes, scores = boxes[mask], classes[mask], scores[mask]

    if boxes.shape[0] == 0:
        return torch.empty((0,6))

    # Clip boxes to tile (keep absolute coordinates)
    cx1 = torch.max(boxes[:,0], torch.tensor(x_tile))
    cy1 = torch.max(boxes[:,1], torch.tensor(y_tile))
    cx2 = torch.min(boxes[:,2], torch.tensor(x_tile2))
    cy2 = torch.min(boxes[:,3], torch.tensor(y_tile2))

    tile_labels = torch.stack([classes.float(), cx1, cy1, cx2, cy2, scores], dim=1)
    return tile_labels

def compute_intersection_area_tensor(box_a, boxes_b):
    """
    box_a: [4] tensor
    boxes_b: [N, 4] tensor
    returns: [N] tensor of intersection areas
    """
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = boxes_b[:,0], boxes_b[:,1], boxes_b[:,2], boxes_b[:,3]
    
    xi1 = torch.max(xa1, xb1)
    yi1 = torch.max(ya1, yb1)
    xi2 = torch.min(xa2, xb2)
    yi2 = torch.min(ya2, yb2)
    
    inter_area = torch.clamp(xi2 - xi1, min=0) * torch.clamp(yi2 - yi1, min=0)
    return inter_area

def pad_to_multiple(image, tile_size=640, pad_value=(114,114,114)):
    h, w = image.shape[:2]
    pad_w = math.ceil(w / tile_size) * tile_size - w
    pad_h = math.ceil(h / tile_size) * tile_size - h
    padded = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=pad_value)
    return padded, w, h  # return original width/height for label conversion

In [66]:

def load_label_tiles(label_dir, filename, tile_size=640, device='cuda'):
    """
    Loads label tiles for a given image and converts YOLO tile-relative coordinates
    to absolute coordinates (xmin, ymin, xmax, ymax) in the full image.

    Returns a list of tensors [M,6] per tile: [class, xmin, ymin, xmax, ymax, score]
    """
    def extract_tile_number(f):
        # Extract tile number from filename like "example_image_tile_3.txt"
        num_part = f.split("_tile_")[-1].replace('.txt','')
        return int(num_part)

    # Get label files for the given image
    label_files = [f for f in os.listdir(label_dir) if f.startswith(os.path.splitext(filename)[0])]
    label_files = sorted(label_files, key=extract_tile_number)

    label_tiles_tensors = []

    for f in label_files:
        tile_idx = extract_tile_number(f)
        # Compute tile position in full image
        x_tile = tile_idx * tile_size  # adjust if you use stride != tile_size
        y_tile = 0                     # modify if tiles are arranged in multiple rows
        tile_box = (x_tile, y_tile, x_tile + tile_size, y_tile + tile_size)

        file_path = os.path.join(label_dir, f)
        with open(file_path, 'r') as file:
            lines = file.read().splitlines()
            if len(lines) > 0:
                # Convert each line: [class, xc, yc, w, h, score] in tile-relative YOLO format
                tile_labels = []
                for line in lines:
                    parts = list(map(float, line.split()))
                    cls, xc, yc, w, h = parts[:5]
                    score = parts[5] if len(parts) > 5 else 1.0  # default score 1.0 if missing
                    # Convert to absolute xyxy coordinates
                    xmin, ymin, xmax, ymax = yolo_tile_to_xyxy_abs([xc, yc, w, h], tile_box, tile_size)
                    tile_labels.append([cls, xmin, ymin, xmax, ymax, score])
                tile_tensor = torch.tensor(tile_labels, dtype=torch.float32, device=device)
            else:
                tile_tensor = torch.empty((0,6), dtype=torch.float32, device=device)
            label_tiles_tensors.append(tile_tensor)

    return label_tiles_tensors


def yolo_tile_to_xyxy_abs(box, tile_box, tile_size):
    """
    Convert YOLO box normalized to a tile into absolute coordinates in full image.

    box: [xc, yc, w, h] normalized to tile
    tile_box: (x_tile, y_tile, x_tile2, y_tile2) in absolute image coords
    tile_size: size of the tile
    Returns: [xmin, ymin, xmax, ymax] in absolute coordinates
    """
    x_tile, y_tile, _, _ = tile_box
    xc, yc, w, h = box

    # Convert normalized tile coordinates to absolute coordinates
    cx_abs = xc * tile_size + x_tile
    cy_abs = yc * tile_size + y_tile
    w_abs = w * tile_size
    h_abs = h * tile_size

    xmin = cx_abs - w_abs / 2
    ymin = cy_abs - h_abs / 2
    xmax = cx_abs + w_abs / 2
    ymax = cy_abs + h_abs / 2

    return [xmin, ymin, xmax, ymax]


In [64]:
def compare_labels_vectorized(
    pred_boxes,
    pred_classes,
    pred_scores,
    gt_boxes,
    gt_classes,
    iou_threshold=0.5,
    containment_threshold=0.9
):
    device = pred_boxes.device

    if pred_boxes.numel() == 0 and gt_boxes.numel() == 0:
        return ([], [], []), ([], [], []), ([], [])

    pred_areas = (pred_boxes[:, 2] - pred_boxes[:, 0]).clamp(min=0) * \
                 (pred_boxes[:, 3] - pred_boxes[:, 1]).clamp(min=0)
    gt_areas = (gt_boxes[:, 2] - gt_boxes[:, 0]).clamp(min=0) * \
               (gt_boxes[:, 3] - gt_boxes[:, 1]).clamp(min=0)

    xx1 = torch.max(pred_boxes[:, None, 0], gt_boxes[None, :, 0])
    yy1 = torch.max(pred_boxes[:, None, 1], gt_boxes[None, :, 1])
    xx2 = torch.min(pred_boxes[:, None, 2], gt_boxes[None, :, 2])
    yy2 = torch.min(pred_boxes[:, None, 3], gt_boxes[None, :, 3])

    w = (xx2 - xx1).clamp(min=0)
    h = (yy2 - yy1).clamp(min=0)
    inter = w * h

    union = pred_areas[:, None] + gt_areas[None, :] - inter
    iou = inter / (union + 1e-6)

    min_area = torch.min(pred_areas[:, None], gt_areas[None, :])
    containment = inter / (min_area + 1e-6)

    class_match = pred_classes[:, None] == gt_classes[None, :]
    match_matrix = class_match & ((iou >= iou_threshold) | (containment >= containment_threshold))

    matched_pred = torch.full((pred_boxes.size(0),), False, device=device)
    matched_gt = torch.full((gt_boxes.size(0),), False, device=device)

    tp_boxes, tp_classes, tp_scores = [], [], []
    fp_boxes, fp_classes, fp_scores = [], [], []

    for i in range(pred_boxes.size(0)):
        possible = torch.where(match_matrix[i] & ~matched_gt)[0]
        if len(possible) > 0:
            j = possible[0]
            tp_boxes.append(pred_boxes[i].cpu().tolist())
            tp_classes.append(int(pred_classes[i].cpu()))
            tp_scores.append(float(pred_scores[i].cpu()))
            matched_gt[j] = True
            matched_pred[i] = True
        else:
            fp_boxes.append(pred_boxes[i].cpu().tolist())
            fp_classes.append(int(pred_classes[i].cpu()))
            fp_scores.append(float(pred_scores[i].cpu()))

    fn_boxes = [gt_boxes[i].cpu().tolist() for i in range(gt_boxes.size(0)) if not matched_gt[i]]
    fn_classes = [int(gt_classes[i].cpu()) for i in range(gt_classes.size(0)) if not matched_gt[i]]

    return (tp_boxes, tp_classes, tp_scores), (fp_boxes, fp_classes, fp_scores), (fn_boxes, fn_classes)

In [6]:
model = YOLO(f"/user/christoph.wald/u15287/insect_pest_detection/2_5_self_training/runs/detect/train/weights/best.pt")

In [7]:
image_dirs = [
        "/user/christoph.wald/u15287/big-scratch/02_splitted_data/train_labeled/SSL/split/images/train",
        "/user/christoph.wald/u15287/big-scratch/02_splitted_data/train_labeled/SSL/split/images/val"
    ]

# tile labels folder
label_dirs = [
    "/user/christoph.wald/u15287/big-scratch/02_splitted_data/train_labeled/SSL/tiles/train/labels",
    "/user/christoph.wald/u15287/big-scratch/02_splitted_data/train_labeled/SSL/tiles/val/labels"
]

#output folder for predictions
output_dir = f"/user/christoph.wald/u15287/insect_pest_detection/2_5_self_training/predictions"
os.makedirs(output_dir, exist_ok=True)

In [75]:
 # output dict structured by FN/FP/TP -> species -> image
json_results = {"FN": {}, "FP": {}, "TP": {}}

In [76]:
image_dir = image_dirs[0]
label_dir = label_dirs[0]
filename = os.listdir(image_dir)[1]
print(filename)

BRAIIM_0003.jpg


In [77]:
image_path = os.path.join(image_dir, filename)
image = cv2.imread(image_path)

In [78]:
boxes, confs, class_ids = sliding_window_prediction(image, model)
print(boxes[0], confs[0], class_ids[0])
print(len(boxes))

tensor([233.7708,  59.0837, 372.9187, 180.9738], device='cuda:0') tensor(0.8137, device='cuda:0') tensor(0., device='cuda:0')
45


In [79]:
boxes, confs, class_ids = nms(boxes, confs, class_ids, iou_threshold=0.4, device=model.device)
print(len(boxes))

22


In [80]:
boxes, confs, class_ids = filter_mostly_contained_boxes(boxes, confs, class_ids, threshold=0.5)
print(len(boxes))

20


In [81]:
pred_tiles_data = get_labels_per_tile_tensor(image, boxes, class_ids, confs)
print(pred_tiles_data)

[tensor([[  0.0000, 233.7708,  59.0837, 372.9187, 180.9738,   0.8137]], device='cuda:0'), tensor([[0.0000e+00, 6.9030e+02, 2.3514e+02, 8.4675e+02, 3.5293e+02, 8.3962e-01]], device='cuda:0'), tensor([], size=(0, 6)), tensor([], size=(0, 6)), tensor([], size=(0, 6)), tensor([], size=(0, 6)), tensor([], size=(0, 6)), tensor([], size=(0, 6)), tensor([], size=(0, 6)), tensor([[0.0000e+00, 4.1434e+03, 1.9491e+02, 4.2478e+03, 2.9615e+02, 4.5668e-01]], device='cuda:0'), tensor([[0.0000e+00, 4.5619e+03, 1.8269e+02, 4.6654e+03, 3.1131e+02, 8.3191e-01],
        [3.0000e+00, 4.7213e+03, 2.8546e+02, 4.7580e+03, 3.1769e+02, 3.1325e-01]], device='cuda:0'), tensor([], size=(0, 6)), tensor([], size=(0, 6)), tensor([], size=(0, 6)), tensor([], size=(0, 6)), tensor([[0.0000e+00, 8.0059e+02, 6.8111e+02, 9.3384e+02, 7.9310e+02, 9.1776e-01]], device='cuda:0'), tensor([[0.0000e+00, 1.3350e+03, 5.5545e+02, 1.5025e+03, 6.8129e+02, 7.6932e-01]], device='cuda:0'), tensor([[0.0000e+00, 1.3350e+03, 5.5545e+02, 1.5

In [82]:
label_tiles_data = load_label_tiles(label_dir, filename)      
print(label_tiles_data)                    

[tensor([[  0.0000, 235.0003,  68.9997, 371.0003, 184.0000,   1.0000]], device='cuda:0'), tensor([[0.0000e+00, 8.9000e+02, 2.3700e+02, 1.0520e+03, 3.5500e+02, 1.0000e+00]], device='cuda:0'), tensor([], device='cuda:0', size=(0, 6)), tensor([], device='cuda:0', size=(0, 6)), tensor([], device='cuda:0', size=(0, 6)), tensor([], device='cuda:0', size=(0, 6)), tensor([], device='cuda:0', size=(0, 6)), tensor([], device='cuda:0', size=(0, 6)), tensor([], device='cuda:0', size=(0, 6)), tensor([[0.0000e+00, 5.9380e+03, 1.9700e+02, 6.0460e+03, 3.0500e+02, 1.0000e+00]], device='cuda:0'), tensor([[0.0000e+00, 6.5600e+03, 1.8300e+02, 6.6690e+03, 3.1200e+02, 1.0000e+00]], device='cuda:0'), tensor([], device='cuda:0', size=(0, 6)), tensor([], device='cuda:0', size=(0, 6)), tensor([], device='cuda:0', size=(0, 6)), tensor([], device='cuda:0', size=(0, 6)), tensor([[0.0000e+00, 9.9610e+03, 2.4200e+02, 1.0099e+04, 3.5900e+02, 1.0000e+00]], device='cuda:0'), tensor([], device='cuda:0', size=(0, 6)), te

In [83]:
pred = pred_tiles_data[0]
print(pred)
label =label_tiles_data[0]
print(label)
pred_boxes = pred[:, 1:5]
pred_classes = pred[:, 0].long()
pred_scores = pred[:, 5]
print(pred_boxes)
gt_boxes = label[:, 1:5]
gt_classes = label[:, 0].long()
print(gt_boxes)

tensor([[  0.0000, 233.7708,  59.0837, 372.9187, 180.9738,   0.8137]], device='cuda:0')
tensor([[  0.0000, 235.0003,  68.9997, 371.0003, 184.0000,   1.0000]], device='cuda:0')
tensor([[233.7708,  59.0837, 372.9187, 180.9738]], device='cuda:0')
tensor([[235.0003,  68.9997, 371.0003, 184.0000]], device='cuda:0')


In [84]:
tp, fp, fn = compare_labels_vectorized(
        pred_boxes, pred_classes, pred_scores, gt_boxes, gt_classes
    )
print(tp)

([[233.77078247070312, 59.0837287902832, 372.9187316894531, 180.9738006591797]], [0], [0.8137310743331909])


In [85]:
results = []
device = 'cuda'

# This is the per-tile loop:
for pred, label in zip(pred_tiles_data, label_tiles_data):
    # pred = predictions[i]   -> predictions for tile i
    # label = labels[i]       -> labels for tile i

    if pred.numel() == 0:
        pred_boxes = torch.empty((0, 4), device='cuda')
        pred_classes = torch.empty((0,), dtype=torch.long, device='cuda')
        pred_scores = torch.empty((0,), device='cuda')
    else:
        pred_boxes = pred[:, 1:5]
        pred_classes = pred[:, 0].long()
        pred_scores = pred[:, 5]

    if label.numel() == 0:
        gt_boxes = torch.empty((0, 4), device=device)
        gt_classes = torch.empty((0,), dtype=torch.long, device=device)
    else:
        gt_boxes = label[:, 1:5]
        gt_classes = label[:, 0].long()


    # Call your function **per tile**
    tp, fp, fn = compare_labels_vectorized(
        pred_boxes, pred_classes, pred_scores, gt_boxes, gt_classes
    )

    results.append((tp, fp, fn))


In [86]:
for i, (tp, fp, fn) in enumerate(results):
    print(f"Tile {i}:")
    print("  TP:", tp)
    print("  FP:", fp)
    print("  FN:", fn)

Tile 0:
  TP: ([[233.77078247070312, 59.0837287902832, 372.9187316894531, 180.9738006591797]], [0], [0.8137310743331909])
  FP: ([], [], [])
  FN: ([], [])
Tile 1:
  TP: ([], [], [])
  FP: ([[690.3004760742188, 235.13690185546875, 846.7450561523438, 352.92706298828125]], [0], [0.8396214842796326])
  FN: ([[890.0003051757812, 237.0, 1052.0003662109375, 355.0]], [0])
Tile 2:
  TP: ([], [], [])
  FP: ([], [], [])
  FN: ([], [])
Tile 3:
  TP: ([], [], [])
  FP: ([], [], [])
  FN: ([], [])
Tile 4:
  TP: ([], [], [])
  FP: ([], [], [])
  FN: ([], [])
Tile 5:
  TP: ([], [], [])
  FP: ([], [], [])
  FN: ([], [])
Tile 6:
  TP: ([], [], [])
  FP: ([], [], [])
  FN: ([], [])
Tile 7:
  TP: ([], [], [])
  FP: ([], [], [])
  FN: ([], [])
Tile 8:
  TP: ([], [], [])
  FP: ([], [], [])
  FN: ([], [])
Tile 9:
  TP: ([], [], [])
  FP: ([[4143.3818359375, 194.9107666015625, 4247.818359375, 296.1534423828125]], [0], [0.4566827416419983])
  FN: ([[5938.0, 197.0003204345703, 6046.0, 305.00030517578125]], [0]

In [87]:
species = filename.split("_")[0]  # extract species from filename
print(species)

BRAIIM


In [88]:
for tp, fp, fn in results:
    # TP
    tp_boxes, tp_classes, tp_scores = tp
    if tp_classes:
        json_results["TP"].setdefault(species, {}).setdefault(filename, [])
        json_results["TP"][species][filename].extend(
            {"prediction": [cls, score] + box} 
            for cls, score, box in zip(tp_classes, tp_scores, tp_boxes)
        )

    # FP
    fp_boxes, fp_classes, fp_scores = fp
    if fp_classes:
        json_results["FP"].setdefault(species, {}).setdefault(filename, [])
        json_results["FP"][species][filename].extend(
            {"prediction": [cls, score] + box} 
            for cls, score, box in zip(fp_classes, fp_scores, fp_boxes)
        )

    # FN
    fn_boxes, fn_classes = fn
    if fn_classes:
        json_results["FN"].setdefault(species, {}).setdefault(filename, [])
        json_results["FN"][species][filename].extend(
            [cls] + box for cls, box in zip(fn_classes, fn_boxes)
        )

In [89]:
json_results

{'FN': {'BRAIIM': {'BRAIIM_0003.jpg': [[0,
     890.0003051757812,
     237.0,
     1052.0003662109375,
     355.0],
    [0, 5938.0, 197.0003204345703, 6046.0, 305.00030517578125],
    [0, 6559.99951171875, 183.0, 6669.0, 312.00030517578125],
    [0, 9961.0, 242.0, 10099.0, 358.99969482421875],
    [0, 11648.0, 251.9996795654297, 11790.0, 385.0],
    [0, 12535.0, 512.0, 12692.0, 640.0],
    [0, 13109.0, 259.00030517578125, 13278.0, 448.0],
    [0, 15642.0, 284.0, 15768.0, 420.0],
    [0, 19957.0, 421.0, 20099.0, 530.0003051757812],
    [0, 21495.0, 72.0, 21652.0, 207.0003204345703],
    [0, 22467.0, 384.00030517578125, 22650.0, 558.0003051757812],
    [0, 23904.0, 499.00030517578125, 24031.0, 640.0],
    [0, 28495.0, 469.0, 28639.0, 587.0],
    [0, 28917.0, -0.00031999999191612005, 29059.0, 89.99967956542969],
    [0, 30542.0, 481.0, 30698.0, 587.0],
    [0, 30742.0, 481.0, 30898.0, 587.0],
    [0, 31195.0, 451.99969482421875, 31334.0, 545.9996948242188],
    [0, 31670.0, 455.000305175

In [ ]:




        print(f"Total missing ground-truth labels: {len(missing_labels)}")
        print(f"Total extra predictions: {len(extra_preds)}")
        sum_missing += len(missing_labels)
        sum_extra += len(extra_preds)
        new_labels[filename] = extra_preds

print("### Total statistics ###")
print(f"{sum_labels} boxes were given.")
print(f"{sum_predictions} boxes were predicted.")
print(f"{sum_missing} labels were missed.")
print(f"{sum_extra} new labels were found.")




conf_true_positives = np.array(conf_true_positives)
print(f"Mean conf of correct predictions {np.round(np.mean(conf_true_positives),2)} with standard deviation {np.round(np.std(conf_true_positives),2)}")
conf_new_labels = np.array(conf_new_labels)
print(f"Mean conf of new predictions {np.round(np.mean(conf_new_labels),2)} with standard deviation {np.round(np.std(conf_new_labels),2)}")

with open(os.path.join(output_dir,'predictions_wo_threshold.json'), 'w') as f:
    json.dump(new_labels, f, indent=4)